# TNP conformer sensitivity -- specialist arm (CPU)

Rebuild all 597 nanobodies with **NanoBodyBuilder2** (the model TNP itself uses) and **IgFold**,
then compute TNP CDR3 compactness on both and on the crystal.

The question is not "which model is more accurate". It is: **how often does prediction error move a
nanobody across a developability decision boundary?**

Boundaries are calibrated on the 597 crystals in this dataset, because TNP's Table 2 thresholds are
published as an image. The antimode 0.961 separates kinked from extended at AUC 0.909 against an
independent alpha/tau backbone rule, so it is a real boundary, not an arbitrary cut.


In [ ]:
import os, sys, json, glob, time, subprocess
import numpy as np
T0=time.time(); WORK='/kaggle/working'
R={'note':'TNP compactness: crystal vs NanoBodyBuilder2 vs IgFold, 597 nanobodies'}
def note(t):
    R['stage']=t; R['elapsed_min']=round((time.time()-T0)/60,1)
    json.dump(R,open(f'{WORK}/tnp_specialists.json','w'),indent=1,default=str)
    print(t, R['elapsed_min'],'min',flush=True)

_ds=glob.glob('/kaggle/input/**/targets_vhh.json',recursive=True)
assert len(_ds)==1, _ds
TG={f"{t['pdb']}{t['H']}":t for t in json.load(open(_ds[0]))}
print(len(TG),'targets',flush=True)

def must(c,w):
    p=subprocess.run(c,shell=True,text=True,capture_output=True)
    if p.returncode: raise RuntimeError(f'{w} (rc={p.returncode}):\n'+(p.stdout+p.stderr)[-2000:])
def sh(c): return subprocess.run(c,shell=True,text=True,capture_output=True)

# Recipe copied verbatim from nbb2-bench, which built 588/588. ImmuneBuilder declares
# neither openmm/pdbfixer (needed by .refine) nor ANARCI (needed by .sequence_checks),
# and ANARCI additionally needs the HMMER binary on PATH.
must(f'{sys.executable} -m pip install -q ImmuneBuilder','install ImmuneBuilder')
must(f'{sys.executable} -m pip install -q openmm pdbfixer biotite','install openmm/pdbfixer/biotite')
sh('apt-get -qq update && apt-get -qq install -y hmmer')
R['hmmer']=sh('which hmmscan').stdout.strip() or 'MISSING'
print('hmmscan:',R['hmmer'],flush=True)
tried=[]
for cmd in (f'{sys.executable} -m pip install -q anarci',
            f'{sys.executable} -m pip install -q git+https://github.com/oxpig/ANARCI.git'):
    p=sh(cmd); ok=sh(f'{sys.executable} -c "import anarci"').returncode==0
    tried.append(f"{cmd.split()[-1]} -> rc={p.returncode} import={'ok' if ok else 'no'}")
    if ok: break
R['anarci_install']=tried; print('\n'.join(tried),flush=True)

# Prove the whole chain works on a REAL sequence before committing to 597.
# Try several: NanoBodyBuilder2 legitimately refuses N-terminally truncated constructs
# (9 of these 597), so a single probe can fail for a scientific reason and tell us
# nothing about the install.
chk=None
for PROBE in sorted(TG)[:6]:
    probe=("from ImmuneBuilder import NanoBodyBuilder2\n"
           "import openmm, anarci\n"
           'NanoBodyBuilder2().predict({"H": "' + TG[PROBE]['hseq'] + '"}).save("/tmp/probe.pdb")\n'
           'print("PROBE_OK", openmm.__version__)\n')
    chk=subprocess.run([sys.executable,'-c',probe],capture_output=True,text=True)
    if 'PROBE_OK' in chk.stdout:
        print('probe target',PROBE,flush=True); break
    print('probe target',PROBE,'refused, trying next',flush=True)
if chk is None or 'PROBE_OK' not in chk.stdout:
    raise RuntimeError('NanoBodyBuilder2 probe failed on all 6 candidates -- install is broken:\n'
                       +(chk.stdout+chk.stderr)[-2500:])
R['openmm_version']=chk.stdout.split('PROBE_OK')[1].strip()
R['probe_atoms']=sum(1 for l in open('/tmp/probe.pdb') if l.startswith('ATOM'))
print(f"probe OK: openmm {R['openmm_version']}, {R['probe_atoms']} atoms",flush=True)
note('installed')


In [ ]:

import numpy as np

def _ang(a,b,c):
    u,v=a-b,c-b
    return float(np.degrees(np.arccos(np.clip(np.dot(u,v)/(np.linalg.norm(u)*np.linalg.norm(v)),-1,1))))

def _dih(a,b,c,d):
    b0,b1,b2=a-b,c-b,d-c; b1=b1/np.linalg.norm(b1)
    v=b0-np.dot(b0,b1)*b1; w=b2-np.dot(b2,b1)*b1
    return float(np.degrees(np.arctan2(np.dot(np.cross(b1,v),w),np.dot(v,w))))

def compactness(ca, span):
    """TNP: n residues in CDR3 / reach; reach = anchors (IMGT 105,117) to furthest loop point."""
    lo,hi=span; loop=ca[lo:hi]; ok=np.isfinite(loop[:,0]); n=int(hi-lo)
    if ok.sum()<3 or not (np.isfinite(ca[lo,0]) and np.isfinite(ca[hi-1,0])): return None,None,n
    base=(ca[lo]+ca[hi-1])/2.0
    reach=float(np.linalg.norm(loop[ok]-base,axis=-1).max())
    return (n/reach if reach>0 else None), reach, n

def kink(ca, span):
    hi=span[1]; idx=[hi-3,hi-2,hi-1,hi]        # IMGT 115,116,117,118
    if idx[0]<0 or idx[3]>=len(ca): return None,None,'na'
    p=ca[idx]
    if not np.isfinite(p[:,0]).all(): return None,None,'na'
    tau=_ang(p[0],p[1],p[2]); alpha=_dih(p[0],p[1],p[2],p[3])
    if 0<alpha<120 and 85<tau<130: lab='kinked'
    elif alpha<-100 and 100<tau<145: lab='extended'
    else: lab='other'
    return alpha,tau,lab

# boundaries derived from the 597 crystals in this dataset (TNP Table 2 is an image)
ANTIMODE, P5, P95 = 0.961, 0.736, 1.294
def band(v):
    if v is None: return 'na'
    return 'amber_low' if v<P5 else 'amber_high' if v>P95 else 'green'
def conf(v):
    return 'na' if v is None else ('kinked' if v>=ANTIMODE else 'extended')

def ca_from_hca(hca):
    return np.array([[np.nan]*3 if r is None or r[1] is None else r[1] for r in hca],float)

def kabsch(P,Q):
    Pm,Qm=P.mean(0),Q.mean(0)
    V,_,W=np.linalg.svd((P-Pm).T@(Q-Qm)); D=np.diag([1,1,np.sign(np.linalg.det(V@W))])
    return V@D@W,Pm,Qm

def cdr3_rmsd(ca, ref, span):
    """framework-aligned CDR3 CA RMSD"""
    L=len(ref); c3=np.zeros(L,bool); c3[span[0]:span[1]]=True
    ok=np.isfinite(ref[:,0])&np.isfinite(ca[:,0])
    fwsel=ok&~c3
    if fwsel.sum()<40: return None
    R,Pm,Qm=kabsch(ca[fwsel],ref[fwsel])
    al=(ca-Pm)@R+Qm
    s=ok&c3
    if s.sum()<3: return None
    return float(np.sqrt(((al[s]-ref[s])**2).sum(-1).mean()))

print('TNP helpers loaded', flush=True)


In [ ]:
import biotite.structure.io.pdb as pdb_io

def ca_from_pdb(path, L):
    """CA coords in file order. Model outputs may be IMGT-numbered (111A/111B), so never do
    index arithmetic on residue ids -- file order is sequence order."""
    a = pdb_io.PDBFile.read(path).get_structure(model=1)
    a = a[(a.atom_name=='CA') & (a.element!='H')]
    if len(a) != L: return None
    return a.coord.astype(float)

from ImmuneBuilder import NanoBodyBuilder2
nb = NanoBodyBuilder2()
os.makedirs('/tmp/nbb2', exist_ok=True)
nbb2, refused = {}, []
t0=time.time()
for i,(tid,t) in enumerate(sorted(TG.items())):
    p=f'/tmp/nbb2/{tid}.pdb'
    try:
        nb.predict({'H': t['hseq']}).save(p)
        c = ca_from_pdb(p, len(t['hseq']))
        if c is None: refused.append(tid+':len'); continue
        nbb2[tid]=c
    except Exception as e:
        refused.append(f'{tid}:{type(e).__name__}')
    if i%100==99: print(' nbb2',i+1,round((time.time()-t0)/60,1),'min',flush=True)
R['nbb2_built']=len(nbb2); R['nbb2_refused']=refused
print('NanoBodyBuilder2:',len(nbb2),'built,',len(refused),'refused',flush=True)
note('nbb2 done')


In [ ]:
igf={}
try:
    must(f'{sys.executable} -m pip install -q igfold','install igfold')
    from igfold import IgFoldRunner
    runner=IgFoldRunner()
    os.makedirs('/tmp/igf',exist_ok=True)
    t0=time.time()
    for i,(tid,t) in enumerate(sorted(TG.items())):
        p=f'/tmp/igf/{tid}.pdb'
        try:
            runner.fold(p, sequences={'H': t['hseq']}, do_refine=True, do_renum=False)
            c=ca_from_pdb(p,len(t['hseq']))
            if c is not None: igf[tid]=c
        except Exception:
            pass
        if i%100==99: print(' igfold',i+1,round((time.time()-t0)/60,1),'min',flush=True)
except Exception as e:
    R['igfold_error']=f'{type(e).__name__}: {e}'
    print('IgFold unavailable, continuing with NanoBodyBuilder2 only:',R.get('igfold_error'),flush=True)
R['igfold_built']=len(igf)
print('IgFold:',len(igf),'built',flush=True)
note('igfold done')


In [ ]:
rows=[]
for tid,t in sorted(TG.items()):
    span=tuple(t['cdr3']); ref=ca_from_hca(t['hca'])
    cc,cr,n = compactness(ref,span)
    row={'tid':tid,'n_res':n,'cry_compact':cc,'cry_reach':cr,
         'cry_band':band(cc),'cry_conf':conf(cc),'cry_kink':kink(ref,span)[2]}
    for name,store in (('nbb2',nbb2),('igfold',igf)):
        ca=store.get(tid)
        if ca is None: continue
        pc,pr,_=compactness(ca,span)
        row[f'{name}_compact']=pc; row[f'{name}_reach']=pr
        row[f'{name}_band']=band(pc); row[f'{name}_conf']=conf(pc)
        row[f'{name}_kink']=kink(ca,span)[2]
        row[f'{name}_cdr3_rmsd']=cdr3_rmsd(ca,ref,span)
        if cc and pc: row[f'{name}_dcompact']=round(pc-cc,4)
    rows.append(row)
R['rows']=rows

def summarise(name):
    s=[r for r in rows if f'{name}_compact' in r and r['cry_compact'] is not None]
    if not s: return None
    d=[abs(r[f'{name}_dcompact']) for r in s]
    rel=[abs(r[f'{name}_dcompact'])/r['cry_compact'] for r in s]
    flip_band=sum(1 for r in s if r[f'{name}_band']!=r['cry_band'])
    flip_conf=sum(1 for r in s if r[f'{name}_conf']!=r['cry_conf'])
    out={'n':len(s),
         'median_abs_dcompact':round(float(np.median(d)),4),
         'median_rel_err_pct':round(100*float(np.median(rel)),2),
         'p95_rel_err_pct':round(100*float(np.percentile(rel,95)),2),
         'band_flip_pct':round(100*flip_band/len(s),2),
         'conf_flip_pct':round(100*flip_conf/len(s),2),
         'median_cdr3_rmsd':round(float(np.median([r[f'{name}_cdr3_rmsd'] for r in s
                                                   if r.get(f'{name}_cdr3_rmsd') is not None])),3)}
    R.setdefault('summary',{})[name]=out
    print(f"  {name:<8} n={out['n']:<4} |dcompact| {out['median_abs_dcompact']:.4f}  "
          f"rel {out['median_rel_err_pct']:.1f}% (p95 {out['p95_rel_err_pct']:.1f}%)  "
          f"amber-band flips {out['band_flip_pct']:.1f}%  kinked/extended flips {out['conf_flip_pct']:.1f}%",flush=True)
    return out

print('\nTNP compactness: predicted vs crystal')
for nm in ('nbb2','igfold'): summarise(nm)
note('done')
print('\nKILL CRITERION: <10% flag change => metrics robust, direction closes.')
